In [1]:
!pip -q install gradio flask joblib

In [2]:
import pandas as pd, glob

water = pd.read_csv(glob.glob('/content/water_potability*.csv')[0])
heart = pd.read_csv(glob.glob('/content/heart_failure*.csv')[0])

print('WATER', water.shape); display(water.head())
print(water.isna().sum()); print(water['Potability'].value_counts())
print('\nHEART', heart.shape); display(heart.head())
print(heart['DEATH_EVENT'].value_counts())

WATER (3276, 10)


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64
Potability
0    1998
1    1278
Name: count, dtype: int64

HEART (299, 13)


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


DEATH_EVENT
0    203
1     96
Name: count, dtype: int64


In [3]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def train(df, target, name):
    X, y = df.drop(columns=[target]), df[target]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    print(f'===== {name} =====')
    print('Accuracy:', round(accuracy_score(yte, pred), 4))
    print(confusion_matrix(yte, pred))
    print(classification_report(yte, pred))
    joblib.dump(pipe, f"{name.replace(' ', '_')}.joblib")
    return {'pipe': pipe, 'cols': list(X.columns), 'defaults': X.median().round(2).to_dict()}

MODELS = {
    'Water Potability': {**train(water, 'Potability', 'Water Potability'),
                         'labels': {0: 'Not Potable', 1: 'Potable'}},
    'Heart Failure':    {**train(heart, 'DEATH_EVENT', 'Heart Failure'),
                         'labels': {0: 'Survived', 1: 'Death Event'}},
}

===== Water Potability =====
Accuracy: 0.5244
[[208 192]
 [120 136]]
              precision    recall  f1-score   support

           0       0.63      0.52      0.57       400
           1       0.41      0.53      0.47       256

    accuracy                           0.52       656
   macro avg       0.52      0.53      0.52       656
weighted avg       0.55      0.52      0.53       656

===== Heart Failure =====
Accuracy: 0.8
[[37  4]
 [ 8 11]]
              precision    recall  f1-score   support

           0       0.82      0.90      0.86        41
           1       0.73      0.58      0.65        19

    accuracy                           0.80        60
   macro avg       0.78      0.74      0.75        60
weighted avg       0.79      0.80      0.79        60



In [6]:
def make_meta(key, df):
    m = MODELS[key]
    meta = []
    for c in m['cols']:
        s = df[c].dropna()
        meta.append(dict(
            name=c,
            binary=set(s.unique()) <= {0, 1},   # yes/no style column
            min=float(round(s.min(), 2)),
            max=float(round(s.max(), 2)),
            default=float(m['defaults'][c]),
        ))
    return meta

DATA = {'Water Potability': water, 'Heart Failure': heart}
META = {k: make_meta(k, DATA[k]) for k in MODELS}
GOOD_IS_ONE = {'Water Potability': True, 'Heart Failure': False}   # is class 1 a "good" outcome?
ICON = {'Water Potability': '💧', 'Heart Failure': '❤️'}

In [7]:
import gradio as gr
gr.close_all()

CSS = """
.gradio-container {max-width: 1100px !important; margin: auto;}
#title {text-align:center; padding: 18px; border-radius: 14px;
        background: linear-gradient(135deg,#4f46e5,#06b6d4); color:white;}
#title h1 {margin:0; font-size: 30px; color:white;}
#title p {margin:4px 0 0; opacity:.9; color:white;}
"""

def result_card(key, p1):
    m = MODELS[key]
    pred = int(p1 >= 0.5)
    good = (pred == 1) == GOOD_IS_ONE[key]
    bg, fg = ('#dcfce7', '#166534') if good else ('#fee2e2', '#991b1b')
    bars = ''
    for i, p in ((0, 1 - p1), (1, p1)):
        bars += f"""
        <div style="text-align:left;margin-top:10px;font-size:13px;color:{fg}">{m['labels'][i]}: <b>{p*100:.1f}%</b></div>
        <div style="height:12px;background:rgba(0,0,0,.1);border-radius:8px;overflow:hidden">
          <div style="width:{p*100:.1f}%;height:100%;background:{fg}"></div></div>"""
    return f"""
    <div style="background:{bg};color:{fg};padding:22px;border-radius:14px;text-align:center">
      <div style="font-size:14px;opacity:.8">Prediction</div>
      <div style="font-size:30px;font-weight:800;margin:4px 0">{m['labels'][pred]}</div>
      <div style="font-size:14px">Confidence: {max(p1, 1-p1)*100:.1f}%</div>
      {bars}
    </div>"""

EMPTY = '<div style="padding:30px;text-align:center;color:#6b7280;border:2px dashed #d1d5db;border-radius:14px">Enter values and click <b>Predict</b></div>'

def build_tab(key):
    m, meta = MODELS[key], META[key]
    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### Input features")
            inputs = []
            for f in meta:
                if f['binary']:
                    comp = gr.Radio([0, 1], value=int(f['default']), label=f['name'])
                else:
                    step = 1 if f['max'] - f['min'] > 50 else round((f['max'] - f['min']) / 200, 4)
                    comp = gr.Slider(f['min'], f['max'], value=f['default'], step=step, label=f['name'])
                inputs.append(comp)
        with gr.Column(scale=2):
            gr.Markdown("### Result")
            out = gr.HTML(EMPTY)
            btn = gr.Button("🔍 Predict", variant="primary", size="lg")
            reset = gr.Button("↺ Reset to defaults")

    def predict(*vals):
        df = pd.DataFrame([vals], columns=m['cols'])
        return result_card(key, float(m['pipe'].predict_proba(df)[0][1]))

    btn.click(predict, inputs, out)
    reset.click(lambda: [(int(f['default']) if f['binary'] else f['default']) for f in meta] + [EMPTY],
                None, inputs + [out])

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=CSS, title="Logistic Regression App") as demo:
    gr.HTML('<div id="title"><h1>🧠 Logistic Regression Predictor</h1><p>Two Kaggle datasets, one model each</p></div>')
    for k in MODELS:
        with gr.Tab(f"{ICON[k]} {k}"):
            build_tab(k)

demo.launch(share=True)

Closing server running on port: 7860


/tmp/ipykernel_2565/2137694696.py:60: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=CSS, title="Logistic Regression App") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ab34b3dcde1e769d5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
from flask import Flask, request, render_template_string
import threading
from google.colab import output

app = Flask(__name__)

PAGE = """
<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Logistic Regression App</title>
<style>
*{box-sizing:border-box}
body{margin:0;font-family:'Segoe UI',Arial,sans-serif;background:linear-gradient(135deg,#4f46e5,#06b6d4);
     min-height:100vh;padding:30px 15px;color:#1f2937}
.wrap{max-width:900px;margin:auto}
h1{color:#fff;text-align:center;margin:0 0 4px;font-size:32px}
.sub{color:#e0f2fe;text-align:center;margin-bottom:22px}
.tabs{display:flex;gap:10px;justify-content:center;margin-bottom:16px;flex-wrap:wrap}
.tab{padding:10px 22px;border-radius:30px;background:rgba(255,255,255,.25);color:#fff;
     text-decoration:none;font-weight:600}
.tab.active{background:#fff;color:#4f46e5}
.card{background:#fff;border-radius:16px;padding:25px;box-shadow:0 10px 30px rgba(0,0,0,.2)}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:16px}
label{font-size:13px;font-weight:600;display:block;margin-bottom:4px}
small{color:#6b7280;font-size:11px}
input,select{width:100%;padding:10px;border:1px solid #d1d5db;border-radius:8px;font-size:14px}
input:focus,select:focus{outline:2px solid #6366f1}
button{margin-top:22px;width:100%;padding:14px;border:0;border-radius:10px;background:#4f46e5;
       color:#fff;font-size:16px;font-weight:700;cursor:pointer}
button:hover{background:#4338ca}
.result{margin-top:22px;padding:22px;border-radius:14px;text-align:center}
.good{background:#dcfce7;color:#166534}.bad{background:#fee2e2;color:#991b1b}
.result h2{margin:4px 0;font-size:30px}
.row{text-align:left;font-size:13px;margin-top:10px}
.bar{height:12px;background:rgba(0,0,0,.1);border-radius:8px;overflow:hidden}
.fill{height:100%;background:currentColor}
</style></head>
<body><div class="wrap">
<h1>🧠 Logistic Regression Predictor</h1>
<div class="sub">Two Kaggle datasets, one model each</div>

<div class="tabs">
{% for k in keys %}
  <a class="tab {{ 'active' if k==ds else '' }}" href="/?ds={{k}}">{{icons[k]}} {{k}}</a>
{% endfor %}
</div>

<div class="card">
<form method="post" action="/?ds={{ds}}">
  <div class="grid">
  {% for f in meta %}
    <div>
      <label>{{f.name}}</label>
      {% if f.binary %}
        <select name="{{f.name}}">
          {% for o in [0,1] %}
          <option value="{{o}}" {{ 'selected' if values[f.name]|float == o else '' }}>{{o}}</option>
          {% endfor %}
        </select>
        <small>0 = No, 1 = Yes</small>
      {% else %}
        <input type="number" step="any" name="{{f.name}}" value="{{values[f.name]}}" required>
        <small>typical range: {{f.min}} to {{f.max}}</small>
      {% endif %}
    </div>
  {% endfor %}
  </div>
  <button type="submit">🔍 Predict</button>
</form>

{% if result %}
  <div class="result {{ 'good' if good else 'bad' }}">
    <div>Prediction</div>
    <h2>{{result}}</h2>
    <div>Confidence: {{conf}}%</div>
    {% for name, pct in probs %}
      <div class="row">{{name}}: <b>{{pct}}%</b></div>
      <div class="bar"><div class="fill" style="width:{{pct}}%"></div></div>
    {% endfor %}
  </div>
{% endif %}
</div></div></body></html>
"""

@app.route('/', methods=['GET', 'POST'])
def index():
    keys = list(MODELS.keys())
    ds = request.args.get('ds', keys[0])
    if ds not in MODELS:
        ds = keys[0]
    m = MODELS[ds]
    values = {f['name']: f['default'] for f in META[ds]}
    result = good = conf = probs = None
    if request.method == 'POST':
        values = {c: float(request.form[c]) for c in m['cols']}
        p = m['pipe'].predict_proba(pd.DataFrame([values], columns=m['cols']))[0]
        pred = int(p[1] >= 0.5)
        result = m['labels'][pred]
        good = (pred == 1) == GOOD_IS_ONE[ds]
        conf = round(max(p) * 100, 1)
        probs = [(m['labels'][i], round(p[i] * 100, 1)) for i in (0, 1)]
    return render_template_string(PAGE, keys=keys, ds=ds, icons=ICON, meta=META[ds],
                                  values=values, result=result, good=good, conf=conf, probs=probs)

threading.Thread(target=lambda: app.run(port=5001, use_reloader=False), daemon=True).start()
output.serve_kernel_port_as_iframe(5001, height=950)

 * Serving Flask app '__main__'


<IPython.core.display.Javascript object>

 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
